# Parameters

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/tutorial/core/02_parameters.ipynb)

Every number that can vary—a mass, a controller gain, a box bound, a cost weight, a noise seed—lives in `.params` under one uniform protocol across all minilink objects.

In [ ]:
# Local conda: minilink already installed. Colab: clone + path + meshcat.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")
    get_ipython().system("pip install -q meshcat")

## 1. The `.params` protocol across minilink

Every parameterized object exposes `.params` as a Python `dict` (or nested `dict` tree on composite objects) and accepts an optional `params=` argument on its equation methods:

| Object | `.params` keys / shape | Functional override call | Copy / update method |
| --- | --- | --- | --- |
| `System` / `DynamicSystem` | Flat `dict[str, float \| ndarray]` (e.g. `{"m": 1.0, "l": 1.0, ...}`) | `sys.f(x, u, t, params=...)`, `sys.h(x, u, t, params=...)` | Mutate `sys.params[k] = v`; call `sys.refresh()` if structure changes |
| `DiagramSystem` / `SampledDiagramSystem` | Nested `{subsys_id: subsys.params}` | `diagram.f(x, u, t, params=...)` | Mutate a block's number in place, `diagram.params["plant"]["m"] = v`; `diagram.params = tree` takes the complete tree, every block named |
| `Set` / `InputSet` | `BoxSet`: `{"lower", "upper"}`; `BallSet`: `{"center", "radius"}`; `SingletonSet`: `{"point"}`; `IntersectionSet`: `{"0": ..., "1": ...}` | `S.margin(z, t=0.0, params=...)`, `S.contains(z, t=0.0, params=...)`, `S.sample(key, n, params=...)` | Mutate `S.params[k] = v` (or `S.upper = ...`, a view); a copy is `copy.deepcopy(S)` then the same edit |
| `Field` | `QuadraticField`: `{"center", "P"}`; `MarginField`: set's `params` | `V.value(x, u=None, t=0.0, params=...)`, `V.gradient(x, u=None, t=0.0, params=...)` | Mutate `V.params[k] = v`; a copy is `copy.deepcopy(V)` then the same edit |
| `Distribution` | `Gaussian`: `{"mean", "std"}` or `{"mean", "cov"}`; `Uniform`: `{"lower", "upper"}`; `Particles`: `{"points"}` | `law.sample(key, n=None, params=...)`, `law.mean_at(params=...)` | Mutate `law.params[k] = v`; a copy is `copy.deepcopy(law)` then the same edit |
| `CostFunction` | `QuadraticCost`: `{"Q", "R", "S", "x_bar", "u_bar"}`; `TimeCost`: `{"x_bar", "radius"}`; `SumCost`: `{term_id: term.params}`; `ScaledCost`: `{"weight", "cost"}` | `cost.g(x, u, t=0.0, params=...)`, `cost.h(x, t=0.0, params=...)`, `cost.integrate(traj, params=...)` | Attribute views (`cost.Q = ...`) or `cost.params = ...` |
| `PlanningProblem` | `{"plant": ..., "cost": ..., "X": ..., "U": ..., "Xf": ..., "x0": ..., "t0": ..., "discount_rate": ...}` | `problem.realize(key, n=None) -> dict` | `problem.replace(**changes) -> PlanningProblem` |

### Reading, mutating, and overriding `sys.params`

On a `System`, `params=None` reads `self.params`. Passing a dictionary `params=...` to `f` or `h` evaluates the equation at those numbers without mutating `self.params`—the mechanism used by JAX `grad` and `vmap`.

In [ ]:
import numpy as np

from minilink import Pendulum

plant = Pendulum()
plant.params

In [ ]:
x = np.array([0.5, 0.0])
u = np.array([0.0])

# Default parameters (l = 1.0)
dx_default = plant.f(x, u)

# Functional override (l = 2.0) without mutating plant.params
dx_override = plant.f(x, u, 0.0, params={**plant.params, "l": 2.0})

dx_default, dx_override, plant.params["l"]

In [ ]:
# In-place mutation changes the system's stored parameters
plant.params["l"] = 2.0
plant.f(x, u)

### Differentiating w.r.t. parameters with JAX

Because `f(x, u, t, params)` reads `params` directly, `jax.jacobian` (or `plant.jacobian("f", "params")`) differentiates $\dot x$ with respect to the entire parameter dictionary at once, returning a dictionary with the same keys:

In [ ]:
import jax
import jax.numpy as jnp

plant = Pendulum()
df_dparams = jax.jacobian(lambda p: plant.f(jnp.array([0.5, 0.0]), jnp.zeros(1), 0.0, p))(
    plant.params
)
{k: v.shape for k, v in df_dparams.items()}, df_dparams["l"]

### Structural parameters and `sys.refresh()`

Dimensions ($n, m, p$) and port layouts are fixed during a run and are not inside `.params`. When a system subclass derives its dimensions from an attribute, call `sys.refresh()` after changing that attribute so the state vector and ports are rebuilt:

In [ ]:
from minilink import System


class PolyGain(System):
    def __init__(self, degree=2):
        self.degree = degree
        super().__init__(0)
        self.add_input_port("u", dim=1)
        self.add_output_port("y", dim=1, function=self.h, dependencies=("u",))
        self.refresh()

    def refresh(self):
        self.params = {"c": np.ones(self.degree + 1)}

    def h(self, x, u, t=0, params=None):
        params = self.params if params is None else params
        c = params["c"]
        powers = u[0] ** np.arange(len(c))
        return np.array([c @ powers])


poly = PolyGain(degree=2)
y_deg2 = poly.h(None, np.array([2.0]))

poly.degree = 4
poly.refresh()
y_deg4 = poly.h(None, np.array([2.0]))

y_deg2, poly.params["c"].shape, y_deg4

## 2. Diagram parameter trees and live views

On a `DiagramSystem`, `.params` is a live nested dictionary `{subsystem_id: subsystem.params}`. Mutating a leaf through `loop.params["plant"]["m"]` mutates the subsystem's own dictionary directly:

In [ ]:
from minilink import Impedance

loop = Impedance() @ plant
loop.params

In [ ]:
loop.params["plant"]["m"] = 1.5
plant.params["m"], loop.subsystems["controller"].params["Kp"]

## 3. `.params` on sets, fields, distributions, and costs

Sets, fields, and distributions hold their numbers in `.params` exactly as a plant does: an edit `S.params["upper"] = ...` moves the set, their named attributes (`box.upper`, `law.mean`) are views of that dictionary, and their methods (`margin`, `value`, `sample`) accept `params=`. A copy at other numbers is Python's `copy.deepcopy`, then the same edit. Costs store their weights and targets in `cost.params` and expose live attribute views (`cost.Q`, `cost.R`, `cost.x_bar`).

In [ ]:
import copy

from minilink import BoxSet, Gaussian, QuadraticCost
from minilink.core import QuadraticField

box = BoxSet([-1.0, -1.0], [1.0, 1.0])
wider_box = copy.deepcopy(box)
wider_box.params["lower"] = np.array([-2.0, -2.0])
wider_box.params["upper"] = np.array([2.0, 2.0])
box.params, wider_box.margin(np.array([1.5, 0.0]))

In [ ]:
field = QuadraticField(np.zeros(2), np.eye(2))
shifted_field = copy.deepcopy(field)
shifted_field.params["center"] = np.array([1.0, 0.0])
field.value(np.array([1.0, 0.0])), shifted_field.value(np.array([1.0, 0.0]))

In [ ]:
law = Gaussian(mean=[0.0, 0.0], std=[0.1, 0.2])
moved_law = copy.deepcopy(law)
moved_law.params["mean"] = np.array([1.0, -1.0])
moved_law.params["std"] = np.array([0.5, 0.5])
law.params, moved_law.mean, moved_law.cov

In [ ]:
cost = QuadraticCost.from_system(plant)
cost.params.keys(), cost.g(np.array([1.0, 0.0]), np.zeros(1))

## 4. Random blocks, stochastic problem trees, and `realize(key, n)`

A random source block (such as `WhiteNoise`) has `sys.is_random == True` and `sys.realize(key) -> dict`, which draws a fresh stream seed from `key`. On a `PlanningProblem`, `problem.params` combines every role (`"plant"`, `"cost"`, `"X"`, `"U"`, `"Xf"`, `"x0"`, `"t0"`, `"discount_rate"`) and holds numbers only; the laws over uncertain numbers are `problem.distributions`, a tree of the same shape, given as `PlanningProblem(..., distributions={"plant": {"m": Uniform(...)}})`:

| Method | Arguments | Output type & shape | Meaning |
| --- | --- | --- | --- |
| `sys.is_random` | — | `bool` | `True` when the system draws random numbers from a seed in `params` |
| `sys.realize(key)` | `key: int \| Generator \| jax.Array \| None` | `dict` | Parameter dict with a fresh stream seed drawn from `key` |
| `problem.drawn_paths()` | — | `list[tuple[str, ...]]` | Paths in `problem.params` that are drawn per trial |
| `problem.realize(key, n=None)` | `key: int \| Generator \| jax.Array \| None`, `n: int \| None = None` | `dict` (same tree as `problem.params`) | Concrete parameter tree: leaf shape `(dim,)` when `n=None`, `(n, dim)` when `n` is given, or means when `key=None` |

In [ ]:
from minilink import PlanningProblem, Uniform, WhiteNoise

noise = WhiteNoise(1, psd=0.05, sample_period=0.01, seed=0)
noise.is_random, noise.realize(42)

In [ ]:
problem = PlanningProblem(
    plant,
    x0=Uniform([-0.2, -0.2], [0.2, 0.2]),
    cost=cost,
    horizon=3.0,
    distributions={"plant": {"m": Uniform([0.8], [1.2])}},
)
problem.drawn_paths()

In [ ]:
# Single realization (n=None): leaves have shape (dim,)
one_draw = problem.realize(0)
one_draw["x0"].shape, one_draw["plant"]["m"].shape, one_draw["x0"], one_draw["plant"]["m"]

In [ ]:
# Batched realization (n=3): drawn leaves have leading batch dimension (3, dim)
batch_draw = problem.realize(0, n=3)
batch_draw["x0"].shape, batch_draw["plant"]["m"].shape, batch_draw["plant"]["m"]

Next, [`03_sets_fields_distributions`](03_sets_fields_distributions.ipynb): the full API of `Set`, `InputSet`, `Field`, and `Distribution`.